# CyberLex: Digital Evidence & Cyber-Law Toolkit for Indian Advocates

A working toolkit for advocates and law firms handling cyber-crime and electronic-evidence matters in India.

| Module | What it does |
|---|---|
| 1. Statute database | BNS 2023, BSA 2023, BNSS 2023 and IT Act 2000 provisions, each with its old IPC / CrPC / Evidence Act equivalent where relevant |
| 2. Custody log | Append-only, **hash-chained** chain-of-custody log. Editing or deleting any past entry is detected |
| 3. Evidence vault | Copies files in, hashes them (SHA-256), locks them read-only, re-verifies on demand |
| 4. Incident mapper | Narrative or category to candidate statutory provisions |
| 5. Admissibility checklist | BSA s.63(4) certificate, hash disclosure, BNSS s.105 / s.176(3) compliance, custody integrity |
| 6. s.63(4) certificate helper | Pre-fills the facts of a certificate from the vault record |
| 7. Case report | Printable HTML + JSON export for the file or the client |

**Runs on plain Python 3.9+. No packages to install.**

> **Important.** This is a drafting and organisation aid for a *qualified advocate*. It is not legal advice and does not replace reading the bare Act. Statute data was checked against public sources on 6 Nov 2026; confirm every section against the official text before it goes into a filing.

In [1]:
import hashlib
import html
import json
import os
import platform
import re
import shutil
import stat
import datetime as dt
from dataclasses import dataclass, asdict
from pathlib import Path

WORKSPACE = Path("cyberlex_cases")
WORKSPACE.mkdir(exist_ok=True)

IST = dt.timezone(dt.timedelta(hours=5, minutes=30))
STATUTES_VERIFIED_ON = "2026-11-06"

DISCLAIMER = (
    "Preliminary working document prepared with the CyberLex toolkit for review by a qualified advocate. "
    "It is not legal advice. Statutory references were checked against public sources on "
    f"{STATUTES_VERIFIED_ON} and must be confirmed against the official text before filing."
)


def now_ist() -> str:
    return dt.datetime.now(IST).isoformat(timespec="seconds")


print("CyberLex ready.", platform.python_version())

CyberLex ready. 3.13.5


## 1. Statute database

India replaced the IPC, CrPC and Indian Evidence Act with the **BNS, BNSS and BSA** (in force from **1 July 2024**). Proceedings that began earlier may still be governed by the old codes, so each entry shows the predecessor section.

`penalty` is filled only where it was confirmed from a source; otherwise it says "See Act text". The IT Act sections below are unchanged by the new codes.

In [2]:
PROVISIONS = {
    # ---- Bharatiya Nyaya Sanhita, 2023 (replaces IPC 1860) ----
    "BNS-111": dict(act="BNS 2023", section="111", title="Organised crime", replaces="New provision (no IPC equivalent)",
                    penalty="See Act text", note="Wide enough to reach organised cyber-fraud syndicates."),
    "BNS-75": dict(act="BNS 2023", section="75", title="Sexual harassment", replaces="IPC 354A", penalty="See Act text", note=""),
    "BNS-77": dict(act="BNS 2023", section="77", title="Voyeurism", replaces="IPC 354C", penalty="See Act text", note=""),
    "BNS-78": dict(act="BNS 2023", section="78", title="Stalking (includes online monitoring)", replaces="IPC 354D", penalty="See Act text", note=""),
    "BNS-316": dict(act="BNS 2023", section="316", title="Criminal breach of trust", replaces="IPC 405-409", penalty="See Act text", note=""),
    "BNS-318": dict(act="BNS 2023", section="318", title="Cheating", replaces="IPC 415-420 (incl. 420)", penalty="See Act text (sub-sections 318(1)-(4) form a punishment ladder)", note=""),
    "BNS-319": dict(act="BNS 2023", section="319", title="Cheating by personation", replaces="IPC 419", penalty="See Act text", note="Fake profiles, impersonation fraud."),
    "BNS-335": dict(act="BNS 2023", section="335", title="Making a false document", replaces="IPC 464", penalty="See Act text", note="Includes false electronic records."),
    "BNS-336": dict(act="BNS 2023", section="336", title="Forgery (incl. forgery of electronic records)", replaces="IPC 463, 465, 468, 469",
                    penalty="336(2) up to 2 yrs; 336(3) forgery for cheating up to 7 yrs; 336(4) forgery to harm reputation up to 3 yrs",
                    note="Some secondary sources number 335-338 inconsistently; check the Act."),
    "BNS-351": dict(act="BNS 2023", section="351", title="Criminal intimidation", replaces="IPC 506", penalty="See Act text", note="Threats via WhatsApp, email, social media."),
    "BNS-356": dict(act="BNS 2023", section="356", title="Defamation", replaces="IPC 499-500", penalty="See Act text", note="Community service added as a punishment option."),
    # ---- Information Technology Act, 2000 ----
    "IT-43": dict(act="IT Act 2000", section="43", title="Unauthorised access, data theft, virus introduction, damage to computer system",
                  replaces="n/a", penalty="Civil: compensation (adjudication)", note="Civil liability; becomes criminal via s.66 if dishonest / fraudulent."),
    "IT-65": dict(act="IT Act 2000", section="65", title="Tampering with computer source documents", replaces="n/a", penalty="Up to 3 yrs and/or fine up to Rs 2 lakh", note=""),
    "IT-66": dict(act="IT Act 2000", section="66", title="Computer-related offences (s.43 acts done dishonestly / fraudulently)", replaces="n/a",
                  penalty="Up to 3 yrs and/or fine up to Rs 5 lakh", note="The usual 'hacking' section in FIRs."),
    "IT-66B": dict(act="IT Act 2000", section="66B", title="Dishonestly receiving stolen computer resource or communication device", replaces="n/a",
                   penalty="Up to 3 yrs and/or fine up to Rs 1 lakh", note=""),
    "IT-66C": dict(act="IT Act 2000", section="66C", title="Identity theft", replaces="n/a", penalty="Up to 3 yrs and fine up to Rs 1 lakh", note=""),
    "IT-66D": dict(act="IT Act 2000", section="66D", title="Cheating by personation using a computer resource", replaces="n/a",
                   penalty="Up to 3 yrs and fine up to Rs 1 lakh", note="Phishing, OTP / UPI fraud."),
    "IT-66E": dict(act="IT Act 2000", section="66E", title="Violation of privacy (private images)", replaces="n/a",
                   penalty="Up to 3 yrs and/or fine up to Rs 2 lakh", note=""),
    "IT-66F": dict(act="IT Act 2000", section="66F", title="Cyber terrorism", replaces="n/a", penalty="Up to imprisonment for life", note=""),
    "IT-67": dict(act="IT Act 2000", section="67", title="Publishing / transmitting obscene material electronically", replaces="n/a",
                  penalty="First conviction: up to 3 yrs and fine up to Rs 5 lakh (higher on repeat)", note=""),
    "IT-67A": dict(act="IT Act 2000", section="67A", title="Publishing / transmitting sexually explicit material electronically", replaces="n/a", penalty="See Act text", note=""),
    "IT-67B": dict(act="IT Act 2000", section="67B", title="Child sexual abuse material", replaces="n/a", penalty="See Act text", note="Handle with strict protocols. Do not copy such material into general case folders."),
    "IT-69": dict(act="IT Act 2000", section="69", title="Directions for interception, monitoring or decryption", replaces="n/a", penalty="See Act text", note="Government power, relevant when interception is challenged."),
    "IT-70": dict(act="IT Act 2000", section="70", title="Protected systems", replaces="n/a", penalty="See Act text", note=""),
    "IT-79A": dict(act="IT Act 2000", section="79A", title="Examiner of Electronic Evidence", replaces="n/a", penalty="n/a",
                   note="SC (May 2026) held Part B of the BSA certificate is not limited to s.79A examiners. See checklist."),
    "IT-66A": dict(act="IT Act 2000", section="66A", title="STRUCK DOWN: offensive messages (Shreya Singhal v UoI, 2015)", replaces="n/a",
                   penalty="Unenforceable", note="Do not cite. Still appears in some FIRs; useful ground for quashing."),
    # ---- Bharatiya Sakshya Adhiniyam, 2023 (replaces Evidence Act 1872) ----
    "BSA-63": dict(act="BSA 2023", section="63", title="Admissibility of electronic records (certificate under s.63(4))", replaces="Evidence Act s.65B",
                   penalty="n/a", note="Certificate in the Schedule: Part A (party) and Part B (expert); hash value disclosure required."),
    # ---- Bharatiya Nagarik Suraksha Sanhita, 2023 (replaces CrPC 1973) ----
    "BNSS-94": dict(act="BNSS 2023", section="94", title="Summons / order to produce documents, electronic communications incl. devices", replaces="CrPC 91", penalty="n/a", note=""),
    "BNSS-105": dict(act="BNSS 2023", section="105", title="Audio-video recording of search and seizure (forward to Magistrate)", replaces="CrPC 100 (independent-witness model)",
                     penalty="n/a", note="Commentary: non-compliance weakens evidentiary weight; unlikely by itself to invalidate the seizure."),
    "BNSS-106": dict(act="BNSS 2023", section="106", title="Police power to seize property", replaces="CrPC 102", penalty="n/a", note=""),
    "BNSS-176": dict(act="BNSS 2023", section="176(3)", title="Forensic evidence collection and videography for offences punishable with 7+ years", replaces="New provision", penalty="n/a", note=""),
    "BNSS-185": dict(act="BNSS 2023", section="185", title="Search by police officer (audio-video recording applies)", replaces="CrPC 165", penalty="n/a", note=""),
}

print(f"{len(PROVISIONS)} provisions loaded (verified {STATUTES_VERIFIED_ON}).")

32 provisions loaded (verified 2026-11-06).


## 2. Tamper-evident chain-of-custody log

Every entry stores the SHA-256 of the previous entry. Change, remove or reorder any earlier line and every later hash stops matching.

One honest limitation: a chain cannot tell that its **last** entries were deleted. So `head` (the latest hash) should be written down somewhere outside the log, for example in the case report, an email to the client, or the covering letter. Pass it back as `expected_head` when verifying.

In [3]:
def _digest(entry: dict) -> str:
    body = {k: v for k, v in entry.items() if k != "entry_hash"}
    canon = json.dumps(body, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return hashlib.sha256(canon.encode("utf-8")).hexdigest()


class CustodyLog:
    GENESIS = "0" * 64

    def __init__(self, path):
        self.path = Path(path)
        self.entries, self.load_errors = [], []
        if self.path.exists():
            with self.path.open(encoding="utf-8") as f:
                for n, line in enumerate(f, start=1):
                    if not line.strip():
                        continue
                    try:
                        self.entries.append(json.loads(line))
                    except json.JSONDecodeError:
                        self.load_errors.append(f"Line {n} is not valid JSON (file damaged or edited).")

    @property
    def head(self) -> str:
        return self.entries[-1]["entry_hash"] if self.entries else self.GENESIS

    def append(self, evidence_id, action, actor, details="", data=None) -> dict:
        entry = {
            "seq": len(self.entries) + 1,
            "timestamp": now_ist(),
            "evidence_id": evidence_id,
            "action": action,
            "actor": actor,
            "details": details,
            "data": data or {},
            "prev_hash": self.head,
        }
        entry["entry_hash"] = _digest(entry)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        with self.path.open("a", encoding="utf-8") as f:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")
        self.entries.append(entry)
        return entry

    def verify(self, expected_head=None):
        problems = list(self.load_errors)
        prev = self.GENESIS
        for i, e in enumerate(self.entries, start=1):
            if e.get("seq") != i:
                problems.append(f"Entry {i}: sequence number is {e.get('seq')} (entry removed or reordered?).")
            if e.get("prev_hash") != prev:
                problems.append(f"Entry {i}: link to previous entry is broken.")
            if _digest(e) != e.get("entry_hash"):
                problems.append(f"Entry {i}: contents do not match its hash (entry edited).")
            prev = e.get("entry_hash")
        if expected_head and prev != expected_head:
            problems.append("Head hash differs from the anchored value (log truncated or extended after anchoring).")
        return (not problems), problems

    def for_evidence(self, evidence_id):
        return [e for e in self.entries if e["evidence_id"] == evidence_id]


print("CustodyLog defined.")

CustodyLog defined.


## 3. Evidence vault

`acquire()` does what a careful examiner does by hand:
1. hash the source,
2. copy it into the vault,
3. hash the copy and refuse to proceed if the two differ,
4. lock the copy read-only,
5. write the hash into the **chained** log, so it cannot be quietly changed in the manifest later.

SHA-256 is the primary hash. MD5 is also recorded purely for cross-checking against older tools that only print MD5. Do not rely on it.

Work on copies for analysis, never on the vault file.

In [4]:
def hash_file(path, chunk=1024 * 1024):
    sha, md5 = hashlib.sha256(), hashlib.md5()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            sha.update(block)
            md5.update(block)
    return sha.hexdigest(), md5.hexdigest()


class EvidenceVault:
    def __init__(self, case_dir, log: CustodyLog):
        self.case_dir = Path(case_dir)
        self.dir = self.case_dir / "vault"
        self.dir.mkdir(parents=True, exist_ok=True)
        self.manifest_path = self.case_dir / "manifest.json"
        self.log = log
        self.manifest = json.loads(self.manifest_path.read_text(encoding="utf-8")) if self.manifest_path.exists() else {}

    def _save(self):
        self.manifest_path.write_text(json.dumps(self.manifest, indent=2, ensure_ascii=False), encoding="utf-8")

    def acquire(self, src, *, description, source_device, collected_by, collected_from="", legal_basis=""):
        src = Path(src)
        if not src.is_file():
            raise FileNotFoundError(f"{src} is not a file")
        eid = f"EV-{len(self.manifest) + 1:03d}"
        sha_src, md5_src = hash_file(src)
        dest = self.dir / f"{eid}__{src.name}"
        shutil.copy2(src, dest)
        sha_dest, _ = hash_file(dest)
        if sha_src != sha_dest:
            dest.unlink()
            self.log.append(eid, "ACQUISITION_FAILED", collected_by, f"Copy hash mismatch for {src.name}")
            raise RuntimeError("Copy does not match source. Acquisition aborted and logged.")
        os.chmod(dest, stat.S_IREAD)
        st = src.stat()
        rec = dict(
            evidence_id=eid, original_name=src.name, vault_file=str(dest.relative_to(self.case_dir)),
            description=description, source_device=source_device, collected_by=collected_by,
            collected_from=collected_from, legal_basis=legal_basis, size_bytes=st.st_size,
            source_modified=dt.datetime.fromtimestamp(st.st_mtime, IST).isoformat(timespec="seconds"),
            sha256=sha_src, md5_reference_only=md5_src, acquired_at=now_ist(),
        )
        self.manifest[eid] = rec
        self._save()
        self.log.append(eid, "ACQUIRED", collected_by,
                        f"{src.name} from {source_device}" + (f" ({legal_basis})" if legal_basis else ""),
                        data={"sha256": sha_src, "size_bytes": st.st_size, "original_name": src.name})
        return rec

    def transfer(self, evidence_id, from_person, to_person, purpose, mode="hand-over"):
        self._need(evidence_id)
        return self.log.append(evidence_id, "TRANSFERRED", from_person,
                               f"To {to_person}. Purpose: {purpose}. Mode: {mode}.",
                               data={"to": to_person})

    def note(self, evidence_id, actor, text, action="NOTE"):
        self._need(evidence_id)
        return self.log.append(evidence_id, action, actor, text)

    def _need(self, evidence_id):
        if evidence_id not in self.manifest:
            raise KeyError(f"Unknown evidence id {evidence_id}")

    def _logged_hash(self, evidence_id):
        for e in self.log.entries:
            if e["evidence_id"] == evidence_id and e["action"] == "ACQUIRED":
                return e["data"].get("sha256")
        return None

    def verify_all(self, actor="CyberLex integrity check"):
        results = []
        for eid, rec in self.manifest.items():
            path = self.case_dir / rec["vault_file"]
            logged = self._logged_hash(eid)
            if not path.exists():
                ok, now = False, "FILE MISSING"
            else:
                now, _ = hash_file(path)
                ok = (now == rec["sha256"] == logged)
            self.log.append(eid, "INTEGRITY_VERIFIED" if ok else "INTEGRITY_FAILURE", actor,
                            f"SHA-256 now {now}", data={"sha256_now": now})
            results.append(dict(evidence_id=eid, ok=ok, sha256_now=now, sha256_recorded=rec["sha256"], sha256_in_log=logged))
        return results


print("EvidenceVault defined.")

EvidenceVault defined.


## 4. Incident-to-statute mapper

Two ways in: pick categories yourself, or paste a client narrative and let `suggest_categories()` propose candidates by keyword. The keyword step is a **lead generator**, not an opinion. The advocate decides what is actually made out on the facts.

Police commonly add BNS and IT Act provisions together in the same FIR, so most categories carry both.

In [5]:
CATEGORIES = {
    "online_fraud": dict(
        name="Online / UPI / phishing fraud",
        keywords=["phishing", "otp", "upi", "refund", "kyc", "fraud", "scam", "cheated", "bank", "card", "link", "lottery", "investment", "fake website"],
        provisions=["BNS-318", "BNS-319", "IT-66C", "IT-66D", "IT-66"],
        evidence=["Chat / SMS / email export with headers", "Bank statement and transaction IDs", "Screenshots of the fake site or link (with URL and timestamp)", "Call detail records (needs s.63 certificate from the nodal officer)"]),
    "identity_theft": dict(
        name="Identity theft / fake profile / impersonation",
        keywords=["fake profile", "impersonat", "identity", "cloned", "morphed", "pretending", "someone else's", "stolen identity", "fake account"],
        provisions=["IT-66C", "IT-66D", "BNS-319", "BNS-351", "BNS-356"],
        evidence=["Profile URL and screenshots with capture time", "Platform notice / takedown correspondence", "Login and IP details obtained via lawful process"]),
    "hacking": dict(
        name="Hacking / unauthorised access / data breach",
        keywords=["hack", "unauthorised access", "unauthorized access", "breach", "password", "compromised", "ransomware", "malware", "virus", "defac", "ddos", "credential"],
        provisions=["IT-43", "IT-66", "IT-65", "IT-70"],
        evidence=["Server / application logs", "Disk or memory image taken by a qualified examiner", "Timeline of access with IP addresses", "CERT-In incident correspondence if reported"]),
    "data_theft": dict(
        name="Insider data theft / source-code theft",
        keywords=["insider", "resigned", "former employee", "source code", "confidential", "copied", "usb", "trade secret", "client list", "exfiltrat", "data theft"],
        provisions=["IT-43", "IT-66", "BNS-316"],
        evidence=["Endpoint / DLP logs", "Email and cloud-sharing audit trail", "Employment contract and NDA", "USB / device connection history"]),
    "threats_blackmail": dict(
        name="Online threats / blackmail / extortion messages",
        keywords=["threat", "blackmail", "extort", "sextortion", "ransom", "leak", "intimidat", "pay or", "will release", "demand money"],
        provisions=["BNS-351", "BNS-356"],
        evidence=["Original messages with sender identifiers", "Payment demand details and wallet / UPI handles", "Platform report receipts"]),
    "stalking_harassment": dict(
        name="Cyber-stalking / online harassment",
        keywords=["stalk", "harass", "repeatedly messag", "following online", "monitoring", "abusive messages", "trolling", "obscene messages"],
        provisions=["BNS-78", "BNS-75", "BNS-351", "IT-67"],
        evidence=["Full message history, not selected screenshots", "Dates and frequency log", "Blocked-account and new-account evidence"]),
    "private_images": dict(
        name="Private images / voyeurism / non-consensual sharing",
        keywords=["private image", "intimate", "nude", "revenge porn", "mms", "hidden camera", "voyeur", "without consent", "morphed photo", "video leaked"],
        provisions=["BNS-77", "IT-66E", "IT-67", "IT-67A"],
        evidence=["URLs of posts (do not circulate the content further)", "Takedown requests and platform replies", "Hash of the file for platform hash-matching"]),
    "obscene_content": dict(
        name="Obscene / sexually explicit content",
        keywords=["obscene", "explicit", "porn", "adult content", "indecent"],
        provisions=["IT-67", "IT-67A", "IT-67B"],
        evidence=["URL and capture record only. Follow court / police protocol for storage"]),
    "forged_records": dict(
        name="Forged electronic records / documents",
        keywords=["forged", "forgery", "fabricated", "fake document", "edited pdf", "doctored", "fake certificate", "fake email", "tampered record"],
        provisions=["BNS-335", "BNS-336", "BNS-318"],
        evidence=["Original and altered versions with metadata", "Document properties and revision history", "Forensic comparison report"]),
    "organised_cybercrime": dict(
        name="Organised cyber-fraud syndicate",
        keywords=["syndicate", "gang", "organised", "organized", "mule account", "call centre", "call center", "multiple victims", "racket"],
        provisions=["BNS-111", "BNS-318", "IT-66D", "IT-66"],
        evidence=["Mule-account and transaction graph", "Linked victim complaints", "Device and SIM clustering from lawful disclosures"]),
    "cyber_terrorism": dict(
        name="Cyber terrorism / attack on critical systems",
        keywords=["cyber terror", "critical infrastructure", "power grid", "national security", "protected system"],
        provisions=["IT-66F", "IT-70", "IT-66"],
        evidence=["Handle through appropriate authorities and counsel with security clearance"]),
}


def suggest_categories(narrative: str, top=3):
    text = narrative.lower()
    scored = []
    for cid, c in CATEGORIES.items():
        hits = [k for k in c["keywords"] if k in text]
        if hits:
            scored.append((len(hits), cid, hits))
    scored.sort(reverse=True)
    return [dict(category=cid, name=CATEGORIES[cid]["name"], matched=hits) for _, cid, hits in scored[:top]]


def map_incident(category_ids):
    seen, out = set(), []
    for cid in category_ids:
        for pid in CATEGORIES[cid]["provisions"]:
            if pid not in seen:
                seen.add(pid)
                out.append(dict(id=pid, **PROVISIONS[pid]))
    return out


def evidence_hints(category_ids):
    return sorted({h for cid in category_ids for h in CATEGORIES[cid]["evidence"]})


print(f"{len(CATEGORIES)} incident categories loaded.")

11 incident categories loaded.


## 5. Admissibility checklist (BSA 2023 and BNSS 2023)

Electronic records come in under **s.63 BSA**, with the certificate in the Schedule under **s.63(4)**. In *Pune Bar Association v Union of India* (W.P.(C) 599/2026, decided 22 May 2026) the Supreme Court upheld s.63(4), including the hash-value requirement, and held that Part B can be signed by a person with expertise in computer science and cyber forensics, not only a s.79A Examiner of Electronic Evidence.

Sources disagree on whether the hash value sits in Part A or Part B of the form. The checklist therefore says "hash value in the certificate". Use the exact Schedule text when you prepare the filing.

Each item has a status of `True`, `False` or `None` (not yet assessed). Items 5 and 6 are filled in automatically from the log and vault.

In [6]:
CHECKLIST_ITEMS = [
    ("code", "Confirm which code governs: events or proceedings before 1 July 2024 may fall under the IPC / CrPC / Evidence Act (s.65B).", "Transition rule"),
    ("source", "Source device or system identified and described (who controlled it, regular use, working order at the time).", "BSA s.63(2)"),
    ("cert_a", "Certificate prepared in the Schedule form, Part A completed by the person producing the record.", "BSA s.63(4)(c)"),
    ("hash", "Hash value of the electronic record computed and stated in the certificate (SHA-256 used by this toolkit; MD5 / SHA-1 / SHA-256 are the recognised options).", "BSA s.63(4), Pune Bar Assn (2026)"),
    ("chain", "Chain of custody log verifies with no breaks. Head hash anchored outside the log.", "Good practice / authenticity"),
    ("integrity", "Vault copy re-hashed and matches the hash recorded at acquisition.", "Good practice / authenticity"),
    ("cert_b", "Part B signed by a person with expertise in computer science and cyber forensics (need not be a s.79A examiner).", "BSA s.63(4), Pune Bar Assn (2026)"),
    ("av_rec", "If seized by police: audio-video recording of the search and seizure made and forwarded to the Magistrate.", "BNSS s.105 / s.185"),
    ("forensic", "If the offence carries 7+ years: forensic collection at the scene and its videography.", "BNSS s.176(3)"),
    ("production", "If obtained by order or notice: the s.94 order / notice is on file.", "BNSS s.94"),
    ("original", "Original preserved untouched. Analysis done on working copies only.", "Good practice"),
    ("third_party", "Telecom / platform records come with a certificate from the nodal officer or authorised signatory.", "BSA s.63(4)"),
]


class AdmissibilityChecklist:
    def __init__(self, evidence_id):
        self.evidence_id = evidence_id
        self.items = {code: dict(code=code, text=text, authority=auth, status=None, note="") for code, text, auth in CHECKLIST_ITEMS}

    def set(self, code, status, note=""):
        self.items[code]["status"] = status
        self.items[code]["note"] = note
        return self

    def mark_not_applicable(self, *codes):
        for c in codes:
            self.items[c].update(status="n/a", note="Not applicable to this evidence")
        return self

    def auto_check(self, vault: EvidenceVault, log: CustodyLog, expected_head=None):
        ok, problems = log.verify(expected_head)
        note = "; ".join(problems) if problems else "Log verified at " + now_ist()
        if ok and not expected_head:
            note += ". WARNING: head hash not yet anchored outside the log, so deletion of the latest entries would not be detected."
        self.set("chain", ok, note)
        res = {r["evidence_id"]: r for r in vault.verify_all()}.get(self.evidence_id)
        if res is not None:
            self.set("integrity", res["ok"], "SHA-256 matches manifest and log" if res["ok"] else "MISMATCH, see custody log")
        return self

    def summary(self):
        counts = {"pass": 0, "fail": 0, "pending": 0, "n/a": 0}
        for it in self.items.values():
            s = it["status"]
            counts["pass" if s is True else "fail" if s is False else "n/a" if s == "n/a" else "pending"] += 1
        return counts

    def show(self):
        icon = {True: "[PASS]", False: "[FAIL]", None: "[ ?  ]", "n/a": "[n/a ]"}
        print(f"Admissibility checklist for {self.evidence_id}")
        for it in self.items.values():
            print(f"  {icon[it['status']]} {it['text']}  ({it['authority']})" + (f"\n         note: {it['note']}" if it["note"] else ""))
        print("  Summary:", self.summary())


print("AdmissibilityChecklist defined.")

AdmissibilityChecklist defined.


## 6. Section 63(4) certificate helper

Fills the **factual particulars** from the vault record so nothing is mistyped, especially the 64-character hash. It is a working draft. Copy the operative wording and signature blocks from the official Schedule in the Gazette / bare Act.

In [7]:
def s63_certificate_draft(rec: dict, *, party_name, party_designation, party_address, forum="", case_no=""):
    return f"""WORKING DRAFT: CERTIFICATE UNDER SECTION 63(4)(c), BHARATIYA SAKSHYA ADHINIYAM, 2023
(Replace the operative wording with the official Schedule text before filing.)

Forum / Case : {forum} {case_no}

PART A  (to be completed by the party producing the record)
I, {party_name}, {party_designation}, of {party_address}, state that:
 1. Electronic record : {rec['description']}
 2. Original file name: {rec['original_name']}  ({rec['size_bytes']} bytes)
 3. Device / system   : {rec['source_device']}
 4. Obtained from     : {rec['collected_from'] or '[to be completed]'}
 5. Date acquired     : {rec['acquired_at']}
 6. HASH VALUE (SHA-256): {rec['sha256']}
 7. Evidence reference: {rec['evidence_id']}

Signature: ______________________      Date: ____________

PART B  (to be completed by the expert)
Name and qualification: __________________________________________
Hash algorithm used   : SHA-256
Hash value computed   : {rec['sha256']}   [expert to confirm independently, do not copy]
Signature: ______________________      Date: ____________
"""


print("Certificate helper defined.")

Certificate helper defined.


## 7. Case report generator

Produces a printable HTML report (open in a browser, then *Print, Save as PDF*) and a machine-readable JSON export. The report shows the custody-chain verification result and the **head hash**. Record that head hash somewhere outside the case folder.

In [8]:
@dataclass
class Case:
    case_id: str
    title: str
    client: str
    advocate: str
    forum: str = ""
    fir_no: str = ""
    incident_date: str = ""
    narrative: str = ""
    opened_on: str = ""


def _table(headers, rows):
    th = "".join(f"<th>{html.escape(h)}</th>" for h in headers)
    body = "".join("<tr>" + "".join(f"<td>{c}</td>" for c in r) + "</tr>" for r in rows)
    return f"<table><thead><tr>{th}</tr></thead><tbody>{body}</tbody></table>"


def build_report(case: Case, vault: EvidenceVault, log: CustodyLog, category_ids, checklists, out_dir=None, expected_head=None):
    e = html.escape
    out_dir = Path(out_dir or vault.case_dir)
    ok, problems = log.verify(expected_head)
    provisions = map_incident(category_ids)

    ev_rows = [[e(r["evidence_id"]), e(r["original_name"]), e(r["description"]), e(r["source_device"]),
                e(r["collected_by"]), e(r["acquired_at"]), f"<code>{e(r['sha256'])}</code>"] for r in vault.manifest.values()]
    log_rows = [[str(x["seq"]), e(x["timestamp"]), e(x["evidence_id"]), e(x["action"]), e(x["actor"]), e(x["details"]),
                 f"<code>{e(x['entry_hash'][:16])}...</code>"] for x in log.entries]
    prov_rows = [[e(p["act"]), e(p["section"]), e(p["title"]), e(p["replaces"]), e(p["penalty"]), e(p["note"])] for p in provisions]
    chk_sections = ""
    sym = {True: "PASS", False: "FAIL", None: "Pending", "n/a": "n/a"}
    for cl in checklists:
        rows = [[sym[i["status"]], e(i["text"]), e(i["authority"]), e(i["note"])] for i in cl.items.values()]
        chk_sections += f"<h3>{e(cl.evidence_id)}</h3>" + _table(["Status", "Requirement", "Authority", "Note"], rows)

    status_html = ('<p class="ok"><b>Custody chain verified.</b> No breaks found.</p>' if ok else
                   '<p class="bad"><b>CUSTODY CHAIN PROBLEMS:</b><br>' + "<br>".join(e(p) for p in problems) + "</p>")
    hints = "".join(f"<li>{e(h)}</li>" for h in evidence_hints(category_ids))
    cats = ", ".join(e(CATEGORIES[c]["name"]) for c in category_ids)

    page = f"""<!doctype html><html lang="en"><head><meta charset="utf-8"><title>{e(case.case_id)} report</title>
<style>
body{{font-family:Georgia,serif;max-width:980px;margin:2rem auto;padding:0 1rem;color:#1b1b1b;line-height:1.45}}
h1{{border-bottom:3px solid #1f3a5f;padding-bottom:.3rem}} h2{{color:#1f3a5f;margin-top:2rem}}
table{{border-collapse:collapse;width:100%;font-size:.85rem;margin:.6rem 0}} th,td{{border:1px solid #bbb;padding:.35rem .5rem;text-align:left;vertical-align:top}}
th{{background:#eef2f7}} code{{font-size:.75rem;word-break:break-all}}
.ok{{background:#e8f5e9;padding:.6rem}} .bad{{background:#fdecea;padding:.6rem}}
.disc{{border:1px solid #999;padding:.6rem;font-size:.85rem;background:#fafafa}}
@media print{{body{{margin:0}} h2{{page-break-after:avoid}}}}
</style></head><body>
<h1>{e(case.title)}</h1>
<p><b>Case ID:</b> {e(case.case_id)} &nbsp; <b>Client:</b> {e(case.client)} &nbsp; <b>Advocate:</b> {e(case.advocate)}<br>
<b>Forum:</b> {e(case.forum)} &nbsp; <b>FIR / complaint no.:</b> {e(case.fir_no)} &nbsp; <b>Incident date:</b> {e(case.incident_date)}<br>
<b>Report generated:</b> {e(now_ist())}</p>
<p class="disc">{e(DISCLAIMER)}</p>
<h2>1. Facts (as instructed)</h2><p>{e(case.narrative) or '[not recorded]'}</p>
<h2>2. Incident classification and candidate provisions</h2>
<p><b>Categories:</b> {cats}</p>
{_table(["Act", "Section", "Subject", "Replaces", "Penalty (where verified)", "Note"], prov_rows)}
<h3>Evidence usually relevant to these categories</h3><ul>{hints}</ul>
<h2>3. Evidence register</h2>
{_table(["ID", "File", "Description", "Source device", "Collected by", "Acquired", "SHA-256"], ev_rows)}
<h2>4. Chain of custody</h2>{status_html}
<p><b>Log head hash (record this outside the case folder):</b><br><code>{e(log.head)}</code></p>
{_table(["#", "Time (IST)", "Evidence", "Action", "By", "Details", "Entry hash"], log_rows)}
<h2>5. Admissibility checklist</h2>{chk_sections}
<p class="disc">{e(DISCLAIMER)}</p></body></html>"""

    html_path = out_dir / f"{case.case_id}_report.html"
    html_path.write_text(page, encoding="utf-8")
    json_path = out_dir / f"{case.case_id}_export.json"
    json_path.write_text(json.dumps(dict(
        case=asdict(case), generated=now_ist(), statutes_verified_on=STATUTES_VERIFIED_ON, categories=category_ids,
        provisions=provisions, evidence=list(vault.manifest.values()), custody_log=log.entries,
        custody_verified=ok, custody_problems=problems, log_head=log.head,
        checklists={c.evidence_id: list(c.items.values()) for c in checklists}), indent=2, ensure_ascii=False), encoding="utf-8")
    return html_path, json_path


print("Report generator defined.")

Report generator defined.


## 8. Worked example (fictional data)

A complete run on an invented UPI-refund scam. Set `RUN_DEMO = False` once you start on real matters.

In [9]:
RUN_DEMO = True


def _rm_tree(p: Path):
    if p.exists():
        for f in p.rglob("*"):
            try:
                os.chmod(f, stat.S_IWRITE | stat.S_IREAD)
            except OSError:
                pass
        shutil.rmtree(p, ignore_errors=True)


if RUN_DEMO:
    demo_in = Path("demo_inputs")
    demo_in.mkdir(exist_ok=True)
    (demo_in / "whatsapp_export.txt").write_text(
        "[05/10/2026, 21:14] +91 90000 00001: Sir your refund is pending, share OTP to confirm\n"
        "[05/10/2026, 21:15] Client: OTP is 000000 (fictional)\n"
        "[05/10/2026, 21:17] +91 90000 00001: Thank you. Amount credited shortly\n", encoding="utf-8")
    (demo_in / "bank_alert_sms.txt").write_text("INR 48,500.00 debited from A/c XX1234 on 05-Oct-26 via UPI. (fictional)\n", encoding="utf-8")

    case = Case(case_id="CASE-DEMO-001", title="Demo: UPI refund-scam complaint", client="A. Sharma (fictional)",
                advocate="Adv. R. Verma (fictional)", forum="Cyber Cell, Lucknow", fir_no="Draft - not yet registered",
                incident_date="2026-10-05",
                narrative="Client received a call about a pending refund, shared an OTP on WhatsApp and Rs 48,500 was debited via UPI. "
                          "The caller used a number that now appears switched off. Client wants to file a complaint and preserve evidence.",
                opened_on=now_ist())
    case_dir = WORKSPACE / case.case_id
    _rm_tree(case_dir)
    case_dir.mkdir(parents=True)

    log = CustodyLog(case_dir / "custody_log.jsonl")
    vault = EvidenceVault(case_dir, log)

    print("Suggested categories:", suggest_categories(case.narrative))
    cats = ["online_fraud"]

    r1 = vault.acquire(demo_in / "whatsapp_export.txt", description="WhatsApp chat export with the caller",
                       source_device="Client's Android phone (Redmi, IMEI redacted)", collected_by="Adv. R. Verma",
                       collected_from="Client, voluntarily produced")
    r2 = vault.acquire(demo_in / "bank_alert_sms.txt", description="Bank debit SMS alert",
                       source_device="Client's Android phone (Redmi, IMEI redacted)", collected_by="Adv. R. Verma",
                       collected_from="Client, voluntarily produced")
    vault.transfer(r1["evidence_id"], "Adv. R. Verma", "Cyber forensic examiner (Dr. K. Rao)", "Independent hashing and Part B certificate")
    vault.note(r1["evidence_id"], "Dr. K. Rao", "Independent SHA-256 matches value in manifest", action="EXAMINER_NOTE")

    cl1 = AdmissibilityChecklist(r1["evidence_id"]).auto_check(vault, log)
    cl1.set("code", True, "Incident in Oct 2026: BNS / BSA / BNSS apply")
    cl1.set("source", True, "Device and owner recorded in manifest")
    cl1.set("hash", True, "SHA-256 recorded in log and manifest")
    cl1.set("cert_a", None, "To be signed by client")
    cl1.set("cert_b", None, "Awaiting Dr. K. Rao")
    cl1.set("original", True, "Vault copy is read-only")
    cl1.mark_not_applicable("av_rec", "forensic", "production", "third_party")
    cl1.show()

    print("\n" + s63_certificate_draft(r1, party_name="A. Sharma", party_designation="Complainant",
                                       party_address="Lucknow (address to be filled)", forum=case.forum, case_no=case.fir_no))
    head = log.head
    html_path, json_path = build_report(case, vault, log, cats, [cl1], expected_head=None)
    print("Report:", html_path)
    print("JSON  :", json_path)
    print("Anchor this head hash outside the folder:", head)

Suggested categories: [{'category': 'online_fraud', 'name': 'Online / UPI / phishing fraud', 'matched': ['otp', 'upi', 'refund']}]
Admissibility checklist for EV-001
  [PASS] Confirm which code governs: events or proceedings before 1 July 2024 may fall under the IPC / CrPC / Evidence Act (s.65B).  (Transition rule)
         note: Incident in Oct 2026: BNS / BSA / BNSS apply
  [PASS] Source device or system identified and described (who controlled it, regular use, working order at the time).  (BSA s.63(2))
         note: Device and owner recorded in manifest
  [ ?  ] Certificate prepared in the Schedule form, Part A completed by the person producing the record.  (BSA s.63(4)(c))
         note: To be signed by client
  [PASS] Hash value of the electronic record computed and stated in the certificate (SHA-256 used by this toolkit; MD5 / SHA-1 / SHA-256 are the recognised options).  (BSA s.63(4), Pune Bar Assn (2026))
         note: SHA-256 recorded in log and manifest
  [PASS] Chain of cu

## 9. Proof that tampering is caught

Two attacks on a throwaway case:
1. someone edits a past custody entry (for example to change who held the device),
2. someone alters a file in the vault.

Both must be detected.

In [10]:
if RUN_DEMO:
    t_dir = WORKSPACE / "CASE-DEMO-TAMPER"
    _rm_tree(t_dir)
    t_dir.mkdir(parents=True)
    tlog = CustodyLog(t_dir / "custody_log.jsonl")
    tvault = EvidenceVault(t_dir, tlog)
    rec = tvault.acquire(Path("demo_inputs") / "bank_alert_sms.txt", description="Bank SMS", source_device="Phone",
                         collected_by="Adv. R. Verma")
    tvault.transfer(rec["evidence_id"], "Adv. R. Verma", "Examiner", "Hashing")
    anchored = tlog.head

    # Attack 1: rewrite the actor in an old entry
    lines = tlog.path.read_text(encoding="utf-8").splitlines()
    forged = json.loads(lines[1])
    forged["actor"] = "Somebody Else"
    lines[1] = json.dumps(forged, ensure_ascii=False)
    forged_path = t_dir / "custody_log_forged.jsonl"
    forged_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    ok, problems = CustodyLog(forged_path).verify()
    print("Attack 1 (edited log entry) detected:", not ok)
    for p in problems:
        print("   -", p)

    # Attack 1b: delete the last entry, then check against the anchored head
    trunc_path = t_dir / "custody_log_truncated.jsonl"
    trunc_path.write_text("\n".join(tlog.path.read_text(encoding="utf-8").splitlines()[:-1]) + "\n", encoding="utf-8")
    print("Attack 1b (deleted last entry) detected only with anchored head:",
          CustodyLog(trunc_path).verify()[0] is True and not CustodyLog(trunc_path).verify(anchored)[0])

    # Attack 2: alter the vault file
    vf = t_dir / rec["vault_file"]
    os.chmod(vf, stat.S_IWRITE | stat.S_IREAD)
    with open(vf, "ab") as f:
        f.write(b" ")
    result = tvault.verify_all()[0]
    print("Attack 2 (altered evidence file) detected:", not result["ok"])
    print("   recorded :", result["sha256_recorded"])
    print("   now      :", result["sha256_now"])

Attack 1 (edited log entry) detected: True
   - Entry 2: contents do not match its hash (entry edited).
Attack 1b (deleted last entry) detected only with anchored head: True
Attack 2 (altered evidence file) detected: True
   recorded : 2623b94e81e8e259fce9d512335fbd8d6c85f3eeee35ef5f20030bb9b14f380e
   now      : f2c35ca98af0ffbd821a1e4355be6a495f93ab5ddf54430dd7119bb0ff2956ba


## 10. Use it on a real matter

Change the values below, set `RUN_DEMO = False` above if you want a clean workspace, and run this cell. Then add evidence with `vault.acquire(...)`, log movements with `vault.transfer(...)`, work through the checklist, and call `build_report(...)`.

```python
case = Case(case_id="2026-017", title="...", client="...", advocate="...", forum="...", fir_no="...",
            incident_date="2026-..-..", narrative="...", opened_on=now_ist())
case_dir = WORKSPACE / case.case_id
log = CustodyLog(case_dir / "custody_log.jsonl")
vault = EvidenceVault(case_dir, log)
suggest_categories(case.narrative)
rec = vault.acquire(r"C:\path\to\export.zip", description="...", source_device="...", collected_by="...")
```

**Practice points**
- Hash at the first moment of contact, ideally on the device owner's premises, and write the hash in a covering note signed by the owner.
- Keep the head hash outside the folder (email it to yourself and the client after each session).
- Keep a physical-device log separately. This toolkit proves file integrity, not who held a phone on a given day, so use `transfer()` every time a device changes hands.
- Never store CSAM or similar material in a case folder. Record URL, time and hash only, and follow the court's protocol.
- For `IT-66A`, remember it was struck down in 2015.

## Appendix: sources used to verify the statute data (checked 6 Nov 2026)

- Pune Bar Association v Union of India, W.P.(C) 599/2026 (decided 22 May 2026): LiveLaw, SCC Online, JSA Prism (June 2026), Verdictum reports on s.63(4) BSA, hash value and who may sign Part B.
- BSA 2023, Schedule (certificate): advocatekhoj.com bare-act text.
- IT Act 2000 offence sections and penalties: lawgist.in, lawbhoomi.com, 3university.io, Kerala Police circular on 2008 amendments.
- BNS 2023 sections and IPC equivalents: lawsection.in conversion table, lawsikho.com (ss.318, 336), Cyril Amarchand Mangaldas client alert.
- BNSS ss.94, 105, 176(3), 185: LiveLaw, Nyaaya, Bar & Bench, iPleaders commentary.

Secondary sources are fine for orientation but are not authority. Check the Gazette text.